Code snippet to group VIIRS composite products in a single Dataset

In [ ]:
import xarray as xr
import glob
import os
from datetime import datetime, timedelta
from observations import EdelweissGrandesRoussesGrid, reprojection_mf_fsc_l3_to_grid

input_folder = "/home/imperatoren/work/edelweiss_assimilation/data/france/snow_cover/composite"
output_file = "/home/imperatoren/work/edelweiss_assimilation/observation_operator/data/grandesrousses/reprojected_from_france_viirs/mf_fsc_l3_grandesrousses_wy_2021_2022.nc"
edelweiss_grandesrousses_grid = EdelweissGrandesRoussesGrid()
viirs_files = glob.glob(f"{input_folder}/*.nc")
viirs_grandesrousses_reprojected = []
for f in viirs_files:
    viirs = xr.open_dataset(f, engine="rasterio", mask_and_scale=False).isel(band=0).drop_vars("band")
    reproj = reprojection_mf_fsc_l3_to_grid(
        meteofrance_snow_cover=viirs.data_vars["snow_cover_fraction"], output_grid=edelweiss_grandesrousses_grid
    )
    t_coord = datetime.strptime(os.path.basename(f)[11:19], "%Y%m%d") + timedelta(hours=12)
    reproj = reproj.expand_dims("time").assign_coords({"time": [t_coord]})
    viirs_grandesrousses_reprojected.append(reproj)


viirs_grandesrousses = xr.concat(viirs_grandesrousses_reprojected, dim="time")
# viirs_grandesrousses.to_netcdf(output_file)